# 第4章 高峰租借量估计与回归对照

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch04-regression-v1`  
**必做：** OLS；Poisson；NB2  
**对象：** 系统小时租借量：次/小时  
**样本：** 共同测试4376小时；无casual/registered特征  
**划分：** 2011训练8645小时；2012上半年验证4358小时；下半年测试4376小时  
**比较：** 同一训练期标准化、删首类别编码；OLS非负截断；NB2的alpha只按验证期选

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/learning-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 分别读取训练、验证和测试输入
测试标签在参数冻结后才用于评价。天气是当小时实测条件，不等于已实现天气未知时的提前预测。


In [ ]:
FILES=['chapters/data/ch04_'+name+'.csv' for name in ['train','validation','test','solution']]
train,valid,test=[pd.read_csv(ROOT/p) for p in FILES[:3]]
print(len(train),len(valid),len(test));display(train.head())
NUMERIC=['temp','atemp','hum','windspeed']
CATEGORICAL=['mnth','hr','holiday','weekday','weathersit']
assert not {'casual','registered','cnt'} & set(NUMERIC+CATEGORICAL)


## 2. 只用训练集拟合预处理
请检查独热编码为什么删除首类别，为什么不同时放入重复表达的season、workingday。若改变输入列，须标明新配置。


In [ ]:
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
transform=ColumnTransformer([('numeric',StandardScaler(),NUMERIC),('calendar',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),CATEGORICAL)])
X=sm.add_constant(transform.fit_transform(train),has_constant='add')
Xv=sm.add_constant(transform.transform(valid),has_constant='add')
Xt=sm.add_constant(transform.transform(test),has_constant='add')
y=train.cnt.to_numpy(float)
assert np.linalg.matrix_rank(X)==X.shape[1]
print('Design matrix:',X.shape)


## 3. 实际拟合OLS、泊松和NB2
修改ALPHAS后只能根据验证期选取，不依据测试误差选参。请解释NB2的条件方差假设。


In [ ]:
ALPHAS=[0.05,0.2,0.5,1.0]
models={'OLS':sm.OLS(y,X).fit(),'Poisson':sm.GLM(y,X,family=sm.families.Poisson()).fit(maxiter=200)}
trials=[]
for alpha in ALPHAS:
    fitted=sm.GLM(y,X,family=sm.families.NegativeBinomial(alpha=alpha)).fit(maxiter=200)
    assert fitted.converged
    trials.append((metrics(valid.cnt,fitted.predict(Xv))['RMSE'],alpha,fitted))
_,chosen_alpha,models['NB2']=min(trials,key=lambda r:(r[0],r[1]))
validation=pd.DataFrame([[a,score] for score,a,_ in trials],columns=['alpha','validation_RMSE'])
display(validation);print('Frozen alpha:',chosen_alpha)


## 4. 在共同4376小时评价
此时才读取测试标签。比较平均偏差与RMSE是否给出相同模型顺序；不要求NB2一定获胜。


In [ ]:
truth=pd.read_csv(ROOT/FILES[3]).set_index('id')
assert set(test.id)==set(truth.index)
actual=truth.loc[test.id,'cnt'].to_numpy(float)
predictions={name:np.maximum(0,m.predict(Xt)) for name,m in models.items()}
scores=metric_table(actual,predictions);display(scores)
periods=[]
for period,hours in [('morning',[7,8,9]),('evening',[16,17,18])]:
    mask=test.hr.isin(hours).to_numpy()
    for name,pred in predictions.items():periods.append({'period':period,'model':name,**metrics(actual[mask],pred[mask])})
peak=pd.DataFrame(periods);display(peak)


## 5. 画出日内形态并导出自己的结果
图为按小时平均，不能代替逐条误差。主结果JSON可在网页“主实验成果自测”导入。


In [ ]:
plt.figure(figsize=(10,4))
for name,values in {'observed':actual,**predictions}.items():
    profile=pd.Series(values).groupby(test.hr.to_numpy()).mean()
    plt.plot(profile.index,profile,label=name)
plt.xlabel('Hour');plt.ylabel('Rentals/hour');plt.legend();save_plot(ROOT,4,'hourly_service')
outputs={'predictions':predictions_table(test.id,predictions),'validation':validation.to_dict('records'),'peak':peak.to_dict('records')}
bundle=primary(ROOT,4,FILES,{'numeric':NUMERIC,'categorical':CATEGORICAL,'alpha_candidates':ALPHAS,'selected_alpha':chosen_alpha},outputs)
csv_file(ROOT/'outputs/ch04/predictions.csv',outputs['predictions']['columns'],outputs['predictions']['rows'])
csv_file(ROOT/'outputs/ch04/peak_errors.csv',peak.columns,peak.values)


## 6. 完成高峰服务量估计报告
引用一个低估较大的时段，说明为何不能将租借量误差直接换算成需要调运的车辆数。


In [ ]:
report(ROOT,4,'共享单车高峰服务量估计报告',{'共同小时数':len(actual),'总体误差':scores.to_string(index=False),'高峰误差':peak.to_string(index=False)},['哪种模型适合本次服务量估计？','早晚高峰的主要问题是什么？','进一步做站点调度还缺什么？'])
